## Setup

In [ ]:
# Install those librairies first
# This command should be used in a terminal :
!pip install xarray "zarr>=3.1.6" imagecodecs fsspec requests aiohttp matplotlib

# Extra dependencies for the streaming download / export blocks 
!pip install "dask[array]" rasterio tqdm

## Initialization

In [ ]:
# Imports
import os
import xarray as xr
from imagecodecs.zarr import register_codecs

# Register image codecs for Zarr
register_codecs()

In [ ]:
URL="https://stac.webservice-energy.org/data/allskyimager/ikarus/"

## Load metadata

In [ ]:
# Access remote data : only metadata is loaded at this stage
ds = xr.open_dataset(URL, engine="zarr")

# Overview of available variables and dimensions
ds

## Load the data

You can load a subset of data for a speicifc variable, from the remote host.


In [ ]:
# Load all HDR images for a single day
images = ds.hdr_image.sel(time="2024-08-15").load()

# The result is a array Dataarray : a 4D cube of data with proper labels on the axes
images.dims

## Animate

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Show an animation
fig, ax = plt.subplots()
im = ax.imshow(images.isel(time=35).values)
ax.axis("off")

def update(t):
    im.set_data(images.isel(time=t))
    ax.set_title(f"Frame {t}")
    return [im]

anim = FuncAnimation(
    fig, update,
    frames=range(35, images.sizes["time"]),
    interval=100, blit=True)

plt.close(fig)
HTML(anim.to_jshtml())

# Download zarr locally 

To download an offline Zip of a subset, you may use [this web form](https://stac.webservice-energy.org/zarr-subset/?url=/data/allskyimager/ikarus/)


## Export images

The following code exports the dataset's image variables as files on disk.
Per-variable settings are listed below.

| Variable | Description | Output | Folder pattern | Compression |
|---|---|---|---|---|
| `raw_sensor` | 10-bit ADU counts (0..1023) of the Sony IMX708 all-sky fisheye CMOS, de-interlaced per band (4 bands), 5 exposure bursts (`bracket`) per 15-min capture; valid 65..1022, 0 outside the imaged hemisphere | TIFF uint16, 4 bands | `output/images/raw_sensor/{bracket}/{date}/{time}.tif` | lossless, ZSTD-9 predictor 2 |
| `radiance` | Calibrated radiance (W/m^2/sr) per band: solid-angle and exposure-weighted average of the RAW counts; NaN outside the imaged hemisphere | TIFF float32, 4 bands | `output/images/radiance/{date}/{time}.tif` | Lossy, 9 bits of mantissa kept. ZSTD-9 predictor 3. Quantization error of 0.08% of relative RSME only |
| `hdr_image` | Demosaiced high-dynamic-range true-color image for display, one per capture, 3 RGB channels | JPEG, quality 95 | `output/images/hdr_image/{date}/{time}.jpg` | JPG, lossly (display product) |

In [ ]:
import os
import itertools
import warnings
import numpy as np
import xarray as xr
import rasterio
from rasterio.errors import NotGeoreferencedWarning

warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)


def _image_vars(ds):
    """Temporal image variables: name -> (x_dim, y_dim, channel_dim)."""
    found = {}
    for name, da in ds.data_vars.items():
        if "time" not in da.dims:
            continue
        xd = next((d for d in da.dims if d.endswith("_x")), None)
        yd = next((d for d in da.dims if d.endswith("_y")), None)
        cd = next((d for d in da.dims if d in ("rgb", "band")), None)
        if xd and yd and cd:
            found[name] = (xd, yd, cd)
    return found


COMPRESSION_PRESETS = {
    "lerc": dict(COMPRESS="LERC_ZSTD", ZSTD_LEVEL=9, MAX_Z_ERROR=0),
    "zstd": dict(COMPRESS="ZSTD", ZSTD_LEVEL=9),
    "lzw": dict(COMPRESS="LZW"),
    "deflate": dict(COMPRESS="DEFLATE"),
}

# Mantissa-zero float: exact under the low-bit mask, outside radiance range,
# stored as a standard GDAL_NODATA tag so masked reads recover it automatically.
_MANTISSA_SENTINEL = np.float32(-32768.0)


def _variable_config(options, dtype):
    """Compact per-variable config -> rasterio creation options.

    The format is inferred from the ``pattern`` extension (``.jpg``/``.jpeg``
    -> JPEG, otherwise TIFF). Compression is inferred when omitted: ZSTD with
    the dtype-appropriate PREDICTOR (3 on float32 — the significance-byte
    reorder that lets ZSTD eat zeroed low mantissa bytes —, 2 horizontal on
    int). For float32 TIFF, ``quant_bits=N`` keeps the N most significant
    mantissa bits (uniform *relative* pixel error, right for wide-dynamic-range
    solar fisheye radiance); NaN is then stored as sentinel under a GDAL_NODATA
    tag, auto-masked on read. ``compression``/``format``/``quality`` can still
    override. Benchmarks at the 0.1% relative-RMSE budget: radiance
    quant_bits=9 ~0.75MB/frame @0.083% vs 1.81MB lossless; raw uint16 lossless
    ZSTD-9 p2 ~0.41MB (LERC-9 0.36MB but needs liblerc downstream).
    """
    pattern = options.get("pattern")
    fmt = (os.path.splitext(pattern)[1].lstrip(".").lower() if pattern
           else str(options.get("format") or "tif").lstrip(".").lower())
    if fmt == "jpg":
        return dict(driver="JPEG", QUALITY=int(options.get("quality", 95)))
    if fmt != "tif":
        raise ValueError(f"unsupported format/extension '{fmt}' (use tif or jpg)")
    comp = options.get("compression")
    if comp is None:
        comp = "zstd"
    elif str(comp).lower() in ("none", ""):
        return dict(driver="GTiff", TILED="YES")
    comp = str(comp).lower()
    if comp not in COMPRESSION_PRESETS:
        raise ValueError(f"unknown compression '{options.get('compression')}' "
                         f"(use {', '.join(COMPRESSION_PRESETS)}, None, or omit)")
    copts = dict(COMPRESSION_PRESETS[comp])
    if comp in ("zstd", "lzw", "deflate"):
        copts.setdefault("PREDICTOR", 3 if np.dtype(dtype).kind == "f" else 2)
    quant = options.get("quant_bits")
    if quant is not None:
        if np.dtype(dtype) != np.float32 or comp != "zstd":
            raise ValueError("quant_bits is only supported for float32 + zstd")
        quant = int(quant)
        if not 0 < quant < 23:
            raise ValueError("quant_bits must be in ]0, 23[")
        copts["_drop_bits"] = 23 - quant
        copts["_nodata"] = _MANTISSA_SENTINEL
    return dict(driver="GTiff", TILED="YES") | copts


def _set_colorinterp(dst):
    """RGB tags so GIS tools recognize true-color rasters."""
    from rasterio.enums import ColorInterp
    if dst.driver != "GTiff" or dst.count < 3:
        return
    ci = list(dst.colorinterp)
    for i, c in enumerate((ColorInterp.red, ColorInterp.green, ColorInterp.blue)):
        ci[i] = c
    dst.colorinterp = ci


def _shard_chunks(da):
    """Chunk dict aligned on the store shards (-1 means full-dimension).

    Aligning on the shards makes dask decode each shard exactly once instead
    of once per frame: 33s vs 142s (4.3x) on this dataset, see benchmark.
    """
    try:
        return {d: (n if n > 0 else da.sizes[d]) for d, n in da.zarr_shards.items()}
    except Exception:
        return {d: da.sizes[d] for d in da.dims}


def _time_range_slice(time_range):
    """Normalize the optional temporal slice to a ``slice`` object.

    Accepted: None (full extent), a ``slice``, a (start, stop) pair, or a
    single start timestamp (open-ended).
    """
    if time_range is None:
        return slice(None)
    if isinstance(time_range, slice):
        return time_range
    if isinstance(time_range, (tuple, list)):
        if len(time_range) == 2:
            return slice(*time_range)
        if len(time_range) == 1:
            return slice(time_range[0], None)
        raise ValueError("time_range must be a slice or a (start, stop) pair")
    return slice(time_range, None)


DATE_FMT, TIME_FMT = "%Y-%m-%d", "%H-%M"  # hardcoded date/time parts


def _layout_pattern(name, options, frame_axes):
    """Folder/file pattern for one variable, from ``options["pattern"]``.

    Returns ``options["pattern"]`` when given (the extension is part of it),
    otherwise a default is built from the ``format`` key:
    ``<name>/<date>/<dim>=<value>/.../<time>.<ext>`` — one explicit folder per
    extra frame dimension (``bracket=2``). Supported placeholders: any
    frame-axis name (e.g. ``{bracket}``), plus the ``{date}``/``{time}`` parts
    formatted with the hardcoded DATE_FMT/TIME_FMT.
    """
    pat = options.get("pattern")
    if pat is not None:
        return pat
    fmt = str(options.get("format") or "tif").lstrip(".").lower()
    dims = "".join(f"/{d}={{{d}}}" for d in frame_axes if d != "time")
    return f"{name}/{{date}}{dims}/{{time}}.{fmt}"


def _block_paths(da, block_idx, frame_axes, dest, pattern):
    """File paths of every frame inside a dask block, in C order.

    ``frame_axes`` are the dimensions holding several frames per block
    (typically time, bracket). Paths are expanded eagerly from the small
    coordinate arrays so the write function stays index-free; ``pattern``
    carries its own extension.
    """
    idx = {d: i for i, d in enumerate(da.dims)}
    ranges = []
    for d in frame_axes:
        k = block_idx[idx[d]]
        st = sum(da.chunks[idx[d]][:k])
        ranges.append(da[d].values[st:st + da.chunks[idx[d]][k]])
    paths = []
    for coords in itertools.product(*ranges):
        vals = {d: str(v) for d, v in zip(frame_axes, coords)}
        t = np.datetime64(vals["time"]).astype("datetime64[s]").item()
        vals.update(date=t.strftime(DATE_FMT), time=t.strftime(TIME_FMT))
        paths.append(os.path.join(dest, pattern.format(**vals)))
    return paths


def _write_block(block, paths, profile):
    """Compress one shard-aligned dask block to one file per frame.

    The block keeps every array dimension; its leading frame axes are
    flattened to match ``paths`` (already in C order). float32 frames may be
    lossy-quantized here (quant_bits) before writing; NaN then maps to the
    sentinel, tagged as GDAL_NODATA for automatic masked reads.
    """
    profile = dict(profile)
    drop = profile.pop("_drop_bits", None)
    nodata = profile.pop("_nodata", None)
    frames = np.asarray(block).reshape(len(paths), *block.shape[-3:])
    for frame, path in zip(frames, paths):
        if drop:
            masked = np.isnan(frame) | (frame == nodata)
            bits = frame.view(np.uint32) & ~np.uint32((1 << drop) - 1)
            frame = np.where(masked, frame, bits.view(np.float32))
            frame = np.where(np.isnan(frame), nodata, frame)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with rasterio.open(path, "w", **profile) as dst:
            dst.write(frame)
            if nodata is not None:
                dst.nodata = nodata
            _set_colorinterp(dst)


def export_images(source, variables=None, dest="./output/images/",
                  time_range=None, workers=4):
    """Write the selected image variables to rasters, in parallel.

    ``source`` is a zarr path/URL (remote store or local download).
    ``variables`` selects the variables and their output through a compact
    config: ``{name: dict(pattern=..., quant_bits=..., compression=...)}`` (see
    _variable_config/_layout_pattern). Format and compression are inferred from
    the pattern's extension and the dtype when omitted; ``{date}``/``{time}``
    are hardcoded (DATE_FMT/TIME_FMT). Default: every image variable as ZSTD
    TIFF. ``time_range``: None, a ``slice``, a (start, stop) pair, or a single
    start timestamp. The threaded dask scheduler is used: rasterio and numpy
    release the GIL, so a full export runs in ~40s on 4 threads.
    """
    import dask
    from dask.diagnostics import ProgressBar

    meta = xr.open_zarr(source, consolidated=False)
    image_vars = _image_vars(meta)
    tr = _time_range_slice(time_range)
    if tr != slice(None):
        meta = meta.sel(time=tr)
    if variables is None:
        variables = {n: dict(format="tif", compression=None) for n in image_vars}
    unknown = [n for n in variables if n not in image_vars]
    if unknown:
        print(f"skipping missing/non-image variable(s): {unknown}")

    tasks = []
    for name, options in variables.items():
        if name not in image_vars:
            continue
        xd, yd, cd = image_vars[name]
        da = xr.open_zarr(source, chunks=_shard_chunks(meta[name]),
                          consolidated=False)[name].sel(time=tr)
        da = da.transpose(..., cd, yd, xd)
        copts = _variable_config(options, da.dtype)
        profile = dict(height=da.sizes[yd], width=da.sizes[xd],
                       count=da.sizes[cd], dtype=np.dtype(da.dtype).name) | copts
        frame_axes = [d for d in da.dims if d not in (cd, yd, xd)]
        pattern = _layout_pattern(name, options, frame_axes)
        blocks = da.data.to_delayed()
        for bi in np.ndindex(blocks.shape):
            paths = _block_paths(da, bi, frame_axes, dest, pattern)
            tasks.append(dask.delayed(_write_block)(blocks[bi], paths, profile))

    print(f"{len(tasks)} shard-aligned block(s), workers={workers}")
    with dask.config.set(scheduler="threads", num_workers=workers):
        with ProgressBar():
            dask.compute(*tasks)

    print(f"Saved in {dest}")




In [ ]:
# Export data as images on disc: lossless sensor, lossy radiance (~0.08% rel. RMSE)
export_images(
    URL,
    
    variables={

        "raw_sensor": dict(
            pattern="raw_sensor/bracket={bracket}/{date}/{time}.tif"),

        "radiance": dict(
            pattern="radiance/{date}/{time}.tif",
            quant_bits=9),

        "hdr_image": dict(                         
            pattern="hdr_image/{date}/{time}.jpg",
        )},

    dest="./output/images/",

    # Optional time range
    time_range=slice("2024", "2025"))